# Pascal VOC 2007+2012 - ConvNeXt Faster R-CNN Selective QAT

Notebook này chạy trên Kaggle để tạo benchmark cho pipeline ConvNeXt selective QAT trên dataset Pascal VOC 2007+2012:

1. Clone repo EchteAI.
2. Tìm dataset Kaggle `vijayabhaskar96/pascal-voc-2007-and-2012`.
3. Convert Pascal VOC XML sang COCO JSON.
4. Train FP32 từng epoch, mỗi epoch tự validation + benchmark 100 ảnh + lưu checkpoint/result.
5. Vẽ biểu đồ hội tụ FP32 theo mAP, loss và latency.
6. Khi FP32 hội tụ hoặc chạm số epoch tối đa, bắt đầu train QAT.
7. Train QAT epoch 1, convert INT8 tạm và benchmark FP32 vs INT8.
8. Resume QAT epoch 2, convert INT8 final và benchmark lại.

Lưu ý: checkpoint SeaDronesSee 6 lớp không dùng trực tiếp được cho Pascal VOC 21 lớp. Notebook này train lại FP32 trên Pascal VOC trước rồi mới bù lỗi QAT.

In [ ]:
# Cell 1 - Cấu hình người dùng
from pathlib import Path

REPO_URL = 'https://github.com/NguyenDucThang-tb/EchteAI.git'
REPO_BRANCH = 'main'
DATASET_HANDLE = 'vijayabhaskar96/pascal-voc-2007-and-2012'

ROOT = Path('/kaggle/working')
WORK = ROOT / 'pascal_voc_convnext_qat'
OUTPUT = WORK / 'checkpoints'
LOGS = WORK / 'logs'
COCO_ROOT = WORK / 'coco'

VARIANT = 'M1'

# Dataset Kaggle dùng để resume/lưu toàn bộ checkpoint + benchmark.
# Dataset này đang chứa fp32_best.pt/fp32_last.pt và có thể chứa qat_last.pt đã train bù lỗi.
CHECKPOINT_DATASET = 'nguyenducthangtb/echteai-pascal-voc-convnext-qat'
CHECKPOINT_INPUT_ROOT = Path('/kaggle/input/datasets/nguyenducthangtb/echteai-pascal-voc-convnext-qat')
RESUME_FROM_CHECKPOINT_DATASET = True
UPLOAD_EVERY_EPOCH = True

# FP32: train từng epoch, sau mỗi epoch validation + benchmark 100 ảnh, vẽ hội tụ.
FP32_MAX_EPOCHS = 10
FP32_MAX_EXTRA_EPOCHS_PER_RUN = 1  # Đổi thành 3/5 nếu muốn một session train thêm nhiều epoch FP32.
FP32_PATIENCE = 2
FP32_MIN_DELTA = 0.002
FORCE_START_QAT = True  # True vì bạn đã có FP32 checkpoint đã train trên dataset.

# QAT bù lỗi: resume từ qat_last.pt nếu dataset đã có; nếu chưa có thì khởi tạo từ fp32_best.pt.
QAT_TOTAL_EPOCHS = 5
QAT_MAX_EXTRA_EPOCHS_PER_RUN = 3
QAT_PATIENCE = 2
QAT_MIN_DELTA = 0.002
BENCHMARK_IMAGES = 100
# TensorRT benchmark chay tren toan bo validation khi None; train/epoch benchmark van dung BENCHMARK_IMAGES.
TENSORRT_BENCHMARK_IMAGES = None
TENSORRT_BENCHMARK_TAG = 'full_val' if TENSORRT_BENCHMARK_IMAGES is None else str(TENSORRT_BENCHMARK_IMAGES)

# Để None nếu muốn chạy toàn bộ VOC. Đặt 500/1000 để smoke test nhanh.
TRAIN_LIMIT = None

# Batch size là per-GPU khi chạy DDP. Kaggle T4 x2 => global batch = batch_size * 2.
FP32_BATCH_SIZE = 2
QAT_BATCH_SIZE = 1

# Pascal VOC ít vật thể nhỏ hơn SeaDronesSee, dùng size vừa phải để Kaggle chạy nhanh hơn.
MODEL_MIN_SIZE = 640
MODEL_TRAIN_MIN_SIZES = [512, 608, 640]
MODEL_MAX_SIZE = 1024

# TensorRT hybrid: backbone chạy TensorRT INT8 engine, phần RPN/ROI còn lại PyTorch CUDA.
# Nếu Kaggle thiếu TensorRT package, bật INSTALL_TENSORRT_IF_MISSING=True rồi restart runtime sau khi cài.
INSTALL_TENSORRT_IF_MISSING = False
TENSORRT_HEIGHT = 640
TENSORRT_WIDTH = 1024
TENSORRT_BATCH_SIZE = 1
TENSORRT_WORKSPACE_MB = 4096

OUTPUT.mkdir(parents=True, exist_ok=True)
LOGS.mkdir(parents=True, exist_ok=True)
COCO_ROOT.mkdir(parents=True, exist_ok=True)
print('WORK:', WORK)
print('OUTPUT:', OUTPUT)
print('CHECKPOINT_DATASET:', CHECKPOINT_DATASET)
print('FP32:', {'max_epochs': FP32_MAX_EPOCHS, 'extra_this_run': FP32_MAX_EXTRA_EPOCHS_PER_RUN})
print('QAT:', {'total_epochs': QAT_TOTAL_EPOCHS, 'extra_this_run': QAT_MAX_EXTRA_EPOCHS_PER_RUN})


In [ ]:
# Cell 2 - Clone repo và cài dependencies
import os
import subprocess
import sys
from pathlib import Path

ROOT = Path('/kaggle/working')
os.chdir(ROOT)
REPO = ROOT / 'EchteAI'
if not REPO.exists():
    print(f'Cloning {REPO_URL} -> {REPO}', flush=True)
    subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, REPO_URL, str(REPO)], check=True, cwd=ROOT)
else:
    print('Repo already exists:', REPO, flush=True)
    subprocess.run(['git', 'fetch', 'origin', REPO_BRANCH], check=False, cwd=REPO)
    subprocess.run(['git', 'checkout', REPO_BRANCH], check=True, cwd=REPO)
    subprocess.run(['git', 'pull', '--rebase', 'origin', REPO_BRANCH], check=False, cwd=REPO)

os.chdir(REPO)
# onnx/onnxscript cần cho export ONNX; TensorRT package chỉ cài nếu bật flag ở Cell 1.
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[coco]',
    'onnx', 'onnxscript', 'onnxsim',
], check=True, cwd=REPO)

if INSTALL_TENSORRT_IF_MISSING:
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q',
        'tensorrt', 'tensorrt-cu12', 'tensorrt-cu12-bindings', 'tensorrt-cu12-libs',
    ], check=True)
    print('TensorRT packages installed. Nếu import TensorRT vẫn lỗi, restart Kaggle runtime rồi chạy lại từ đầu.')

import torch
print('Repo:', REPO)
subprocess.run(['git', 'log', '-1', '--oneline'], check=False, cwd=REPO)
print('torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('GPU count:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f'GPU {i}:', torch.cuda.get_device_name(i))
try:
    import tensorrt as trt
    print('TensorRT:', trt.__version__)
except Exception as error:
    print('TensorRT import failed:', repr(error))
    print('Vẫn train/convert PyTorch được. Muốn build .engine thì cần cài TensorRT hoặc dùng môi trường có TensorRT.')


In [ ]:
# Cell 3 - Helper chạy lệnh và đọc checkpoint
import datetime
import json
import os
import subprocess
import sys
from pathlib import Path

import torch

os.chdir(REPO)

def run_and_log(command, log_path, cwd=REPO):
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    env.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print('Command:', ' '.join(map(str, command)), flush=True)
    print('Persistent log:', log_path, flush=True)
    print('Started:', datetime.datetime.now().isoformat(timespec='seconds'), flush=True)
    with log_path.open('a', encoding='utf-8') as log_file:
        log_file.write(f'\n===== START {datetime.datetime.now().isoformat()} =====\n')
        log_file.write(' '.join(map(str, command)) + '\n')
        log_file.flush()
        process = subprocess.Popen(
            [str(x) for x in command],
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            cwd=str(cwd),
            env=env,
        )
        for line in process.stdout:
            print(line, end='', flush=True)
            log_file.write(line)
            log_file.flush()
        code = process.wait()
        log_file.write(f'===== END code={code} {datetime.datetime.now().isoformat()} =====\n')
    if code != 0:
        raise subprocess.CalledProcessError(code, command)


def checkpoint_epoch(path):
    path = Path(path)
    if not path.exists():
        return 0
    payload = torch.load(path, map_location='cpu', weights_only=False)
    return int(payload.get('epoch', 0)) if isinstance(payload, dict) else 0


def checkpoint_size_mb(path):
    path = Path(path)
    return path.stat().st_size / 2**20 if path.exists() else 0.0


def print_checkpoint_summary(path):
    path = Path(path)
    for item in sorted(path.glob('*')):
        if item.is_file():
            print(f'{item.name:35s} {item.stat().st_size / 2**20:8.2f} MB')

In [ ]:
# Cell 4 - Tìm hoặc tải dataset Pascal VOC trên Kaggle
import os
from pathlib import Path

import kagglehub


def find_dataset_root():
    candidates = [
        Path('/kaggle/input/pascal-voc-2007-and-2012'),
        Path('/kaggle/input') / DATASET_HANDLE.split('/')[-1],
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    print('Dataset not attached in /kaggle/input; downloading with kagglehub...', flush=True)
    return Path(kagglehub.dataset_download(DATASET_HANDLE))

DATA_DOWNLOAD = find_dataset_root()
print('DATA_DOWNLOAD:', DATA_DOWNLOAD)
print('Top-level files/folders:')
for item in sorted(DATA_DOWNLOAD.iterdir())[:30]:
    print(' ', item)

In [ ]:
# Cell 5 - Convert Pascal VOC XML sang COCO JSON
import json
import random
import shutil
import xml.etree.ElementTree as ET
from pathlib import Path
from PIL import Image

VOC_CLASSES = [
    'aeroplane', 'bicycle', 'bird', 'boat', 'bottle',
    'bus', 'car', 'cat', 'chair', 'cow',
    'diningtable', 'dog', 'horse', 'motorbike', 'person',
    'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor',
]
CLASS_TO_ID = {name: i + 1 for i, name in enumerate(VOC_CLASSES)}


def find_voc_roots(root):
    roots = []
    for ann_dir in root.rglob('Annotations'):
        voc_root = ann_dir.parent
        if (voc_root / 'JPEGImages').exists():
            roots.append(voc_root)
    roots = sorted(set(roots))
    assert roots, f'Không tìm thấy thư mục Pascal VOC trong {root}'
    return roots


def read_split(voc_root, split):
    split_file = voc_root / 'ImageSets' / 'Main' / f'{split}.txt'
    if not split_file.exists():
        return []
    return [line.strip().split()[0] for line in split_file.read_text().splitlines() if line.strip()]


def collect_records(voc_roots):
    train_records, val_records, all_records = [], [], []
    for voc_root in voc_roots:
        year = voc_root.name
        train_ids = read_split(voc_root, 'trainval') or read_split(voc_root, 'train')
        val_ids = read_split(voc_root, 'test') or read_split(voc_root, 'val')
        ann_dir = voc_root / 'Annotations'
        image_dir = voc_root / 'JPEGImages'
        if not train_ids:
            train_ids = [p.stem for p in ann_dir.glob('*.xml')]
        for image_id in train_ids:
            xml = ann_dir / f'{image_id}.xml'
            jpg = image_dir / f'{image_id}.jpg'
            if xml.exists() and jpg.exists():
                train_records.append((year, image_id, xml, jpg))
                all_records.append((year, image_id, xml, jpg))
        for image_id in val_ids:
            xml = ann_dir / f'{image_id}.xml'
            jpg = image_dir / f'{image_id}.jpg'
            if xml.exists() and jpg.exists():
                val_records.append((year, image_id, xml, jpg))
                all_records.append((year, image_id, xml, jpg))
    if not val_records:
        random.seed(42)
        all_unique = sorted(set(all_records), key=lambda x: (x[0], x[1]))
        random.shuffle(all_unique)
        cut = max(1, int(0.1 * len(all_unique)))
        val_records = all_unique[:cut]
        train_records = all_unique[cut:]
    return train_records, val_records


def voc_record_to_coco(records, image_root, output_json):
    images, annotations = [], []
    ann_id = 1
    image_id = 1
    for year, stem, xml_path, image_path in records:
        try:
            with Image.open(image_path) as img:
                width, height = img.size
        except Exception:
            root = ET.parse(xml_path).getroot()
            size = root.find('size')
            width = int(size.findtext('width'))
            height = int(size.findtext('height'))
        rel_file = image_path.relative_to(image_root).as_posix()
        images.append({'id': image_id, 'file_name': rel_file, 'width': width, 'height': height})
        root = ET.parse(xml_path).getroot()
        for obj in root.findall('object'):
            name = obj.findtext('name')
            difficult = int(obj.findtext('difficult') or 0)
            if name not in CLASS_TO_ID:
                continue
            box = obj.find('bndbox')
            xmin = max(0.0, float(box.findtext('xmin')) - 1.0)
            ymin = max(0.0, float(box.findtext('ymin')) - 1.0)
            xmax = min(float(width), float(box.findtext('xmax')))
            ymax = min(float(height), float(box.findtext('ymax')))
            w = max(0.0, xmax - xmin)
            h = max(0.0, ymax - ymin)
            if w <= 0 or h <= 0:
                continue
            annotations.append({
                'id': ann_id,
                'image_id': image_id,
                'category_id': CLASS_TO_ID[name],
                'bbox': [xmin, ymin, w, h],
                'area': w * h,
                'iscrowd': 0,
                'difficult': difficult,
            })
            ann_id += 1
        image_id += 1
    categories = [{'id': i + 1, 'name': name, 'supercategory': 'object'} for i, name in enumerate(VOC_CLASSES)]
    payload = {'images': images, 'annotations': annotations, 'categories': categories}
    output_json = Path(output_json)
    output_json.parent.mkdir(parents=True, exist_ok=True)
    output_json.write_text(json.dumps(payload), encoding='utf-8')
    return payload

voc_roots = find_voc_roots(DATA_DOWNLOAD)
print('VOC roots:')
for root in voc_roots:
    print(' ', root)
train_records, val_records = collect_records(voc_roots)
IMAGE_ROOT = DATA_DOWNLOAD
TRAIN_JSON = COCO_ROOT / 'instances_train.json'
VAL_JSON = COCO_ROOT / 'instances_val.json'
train_coco = voc_record_to_coco(train_records, IMAGE_ROOT, TRAIN_JSON)
val_coco = voc_record_to_coco(val_records, IMAGE_ROOT, VAL_JSON)
print('Train images:', len(train_coco['images']), 'annotations:', len(train_coco['annotations']))
print('Val images:', len(val_coco['images']), 'annotations:', len(val_coco['annotations']))
print('IMAGE_ROOT:', IMAGE_ROOT)
print('TRAIN_JSON:', TRAIN_JSON)
print('VAL_JSON:', VAL_JSON)

In [ ]:
# Cell 6 - Tạo runtime config cho Pascal VOC
import yaml
from pathlib import Path

base = yaml.safe_load(Path('configs/seadronessee_colab.yaml').read_text())
base['dataset'].update({
    'train_images': str(IMAGE_ROOT),
    'train_annotations': str(TRAIN_JSON),
    'val_images': str(IMAGE_ROOT),
    'val_annotations': str(VAL_JSON),
    'test_images': str(IMAGE_ROOT),
    'test_annotations': str(VAL_JSON),
    'ignore_category_ids': [],
    'num_classes': 21,
    'workers': 2,
})
base['model'].update({
    'backbone': 'convnext_tiny',
    'pretrained_backbone': True,
    'trainable_backbone_layers': 4,
    'min_size': MODEL_MIN_SIZE,
    'train_min_sizes': MODEL_TRAIN_MIN_SIZES,
    'max_size': MODEL_MAX_SIZE,
    'anchor_sizes': 'auto',
    'anchor_statistics_min_size': MODEL_MIN_SIZE,
})
base['training'].update({
    'fp32_batch_size': FP32_BATCH_SIZE,
    'qat_batch_size': QAT_BATCH_SIZE,
    'fp32_epochs': FP32_MAX_EPOCHS,
    'qat_epochs': QAT_TOTAL_EPOCHS,
    'epoch_benchmark_images': BENCHMARK_IMAGES,
    'print_frequency': 50,
    'warmup_iterations': 500,
})
# QAT 2 epoch: epoch 1 full fake-quant, epoch 2 frozen observer để convert final.
base['quantization']['variant'] = VARIANT
base['quantization']['backend'] = 'auto'
base['quantization']['calibration_images'] = 256
base['quantization']['weight_only_warmup_epochs'] = 0
base['quantization']['observer_freeze_epochs'] = 1
base.setdefault('quantization', {}).setdefault('compiler', {})
base['quantization']['compiler'].update({
    'scope': 'backbone',
    'artifact_dir': str(OUTPUT / 'tensorrt_artifacts'),
    'example_batch_size': TENSORRT_BATCH_SIZE,
    'example_height': TENSORRT_HEIGHT,
    'example_width': TENSORRT_WIDTH,
})
base['output'] = {
    'directory': str(OUTPUT),
    'fp32_best': str(OUTPUT / 'fp32_best.pt'),
    'fp32_last': str(OUTPUT / 'fp32_last.pt'),
    'qat_best': str(OUTPUT / 'qat_best.pt'),
    'qat_last': str(OUTPUT / 'qat_last.pt'),
    'int8_model': str(OUTPUT / 'selective_int8.pt'),
    'evaluation_json': str(OUTPUT / 'evaluation.json'),
    'benchmark_json': str(OUTPUT / 'benchmark.json'),
    'epoch_benchmarks': str(OUTPUT / 'epoch_benchmarks.json'),
}
base['benchmark'] = {'warmup_iterations': 10, 'iterations': 50, 'num_threads': 1}
RUNTIME_CONFIG = WORK / 'runtime_pascal_voc.yaml'
RUNTIME_CONFIG.write_text(yaml.safe_dump(base, sort_keys=False), encoding='utf-8')
print('Runtime config:', RUNTIME_CONFIG)
print(RUNTIME_CONFIG.read_text())


In [ ]:
# Cell 7 - Resume checkpoint dataset + train FP32 từng epoch, benchmark 100 ảnh, vẽ hội tụ
import json
import shutil
from pathlib import Path

import kagglehub
import matplotlib.pyplot as plt
import torch

FP32_HISTORY_JSON = OUTPUT / 'fp32_convergence_history.json'
FP32_CONVERGED_FLAG = OUTPUT / 'fp32_converged.json'


def read_json(path, default):
    path = Path(path)
    return json.loads(path.read_text()) if path.exists() else default


def copy_dataset_files_to_output(dataset_handle=CHECKPOINT_DATASET):
    if not RESUME_FROM_CHECKPOINT_DATASET:
        print('RESUME_FROM_CHECKPOINT_DATASET=False; bỏ qua tải checkpoint dataset.')
        return None
    try:
        ckpt_dir = Path(kagglehub.dataset_download(dataset_handle, force_download=True))
    except Exception as error:
        print('Không tải được checkpoint dataset, sẽ train từ trạng thái hiện tại:', repr(error))
        return None
    print('Downloaded checkpoint dataset:', ckpt_dir)
    OUTPUT.mkdir(parents=True, exist_ok=True)
    copied = 0
    for item in ckpt_dir.rglob('*'):
        if item.is_file():
            shutil.copy2(item, OUTPUT / item.name)
            copied += 1
    print(f'Copied {copied} file(s) into OUTPUT:', OUTPUT)
    return ckpt_dir


def read_checkpoint_metrics(path):
    path = Path(path)
    if not path.exists():
        return {}
    payload = torch.load(path, map_location='cpu', weights_only=False)
    return payload.get('metrics', {}) if isinstance(payload, dict) else {}


def fp32_history_record(epoch):
    metrics = read_checkpoint_metrics(OUTPUT / 'fp32_last.pt')
    train = metrics.get('train', {})
    benchmark = metrics.get('benchmark', {})
    return {
        'epoch': int(epoch),
        'train_loss': train.get('loss'),
        'map_50_95': metrics.get('map_50_95'),
        'map_50': metrics.get('map_50'),
        'precision': metrics.get('precision'),
        'recall': metrics.get('recall'),
        'accuracy': metrics.get('accuracy'),
        'mean_iou': metrics.get('mean_iou'),
        'benchmark_latency_ms': benchmark.get('latency_ms_per_image'),
        'benchmark_fps': benchmark.get('fps'),
    }


def update_fp32_history(epoch):
    history = read_json(FP32_HISTORY_JSON, [])
    history = [item for item in history if int(item.get('epoch', -1)) != int(epoch)]
    history.append(fp32_history_record(epoch))
    history = sorted(history, key=lambda item: item['epoch'])
    FP32_HISTORY_JSON.write_text(json.dumps(history, indent=2), encoding='utf-8')
    return history


def fp32_convergence_status(history, patience=2, min_delta=0.002):
    valid = [item for item in history if item.get('map_50_95') is not None]
    if not valid:
        return False, 'no validation metric yet'
    best = -1.0
    stale = 0
    best_epoch = valid[0]['epoch']
    for item in valid:
        value = float(item['map_50_95'])
        if value > best + float(min_delta):
            best = value
            best_epoch = int(item['epoch'])
            stale = 0
        else:
            stale += 1
    converged = stale >= int(patience)
    reason = f'best_epoch={best_epoch} best_map={best:.4f} stale_epochs={stale}/{patience}'
    return converged, reason


def run_fp32_one_epoch():
    fp32_last = OUTPUT / 'fp32_last.pt'
    if torch.cuda.device_count() >= 2:
        command = [
            sys.executable, '-m', 'torch.distributed.run', '--standalone', '--nproc_per_node=2',
            'scripts/train_fp32_ddp.py', '--config', str(RUNTIME_CONFIG), '--epochs-this-run', '1',
            '--no-find-unused-parameters',
        ]
        if fp32_last.exists():
            command += ['--resume', str(fp32_last)]
    else:
        command = [sys.executable, '-u', 'scripts/train_fp32.py', '--config', str(RUNTIME_CONFIG), '--epochs-this-run', '1']
        if fp32_last.exists():
            command += ['--resume', str(fp32_last)]
    if TRAIN_LIMIT is not None:
        command += ['--limit', str(TRAIN_LIMIT)]
    run_and_log(command, LOGS / 'fp32_train_until_converged.log', cwd=REPO)


def upload_dataset(version_notes):
    if not UPLOAD_EVERY_EPOCH:
        print('UPLOAD_EVERY_EPOCH=False; bỏ qua upload.')
        return
    print('Uploading to:', CHECKPOINT_DATASET)
    print_checkpoint_summary(OUTPUT)
    kagglehub.dataset_upload(CHECKPOINT_DATASET, str(OUTPUT), version_notes=version_notes)
    print('Uploaded:', f'https://www.kaggle.com/datasets/{CHECKPOINT_DATASET}')


def plot_fp32_history():
    history = read_json(FP32_HISTORY_JSON, [])
    if not history:
        print('Chưa có FP32 history để vẽ.')
        return
    epochs = [item['epoch'] for item in history]
    map_50_95 = [item.get('map_50_95') for item in history]
    map_50 = [item.get('map_50') for item in history]
    losses = [item.get('train_loss') for item in history]
    latencies = [item.get('benchmark_latency_ms') for item in history]
    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    axes[0].plot(epochs, map_50_95, marker='o', label='mAP@50:95')
    axes[0].plot(epochs, map_50, marker='o', label='mAP@50')
    axes[0].set_title('FP32 validation mAP')
    axes[0].set_xlabel('Epoch')
    axes[0].grid(True)
    axes[0].legend()
    axes[1].plot(epochs, losses, marker='o', color='tab:red')
    axes[1].set_title('FP32 train loss')
    axes[1].set_xlabel('Epoch')
    axes[1].grid(True)
    axes[2].plot(epochs, latencies, marker='o', color='tab:green')
    axes[2].set_title(f'FP32 benchmark latency ({BENCHMARK_IMAGES} ảnh)')
    axes[2].set_xlabel('Epoch')
    axes[2].set_ylabel('ms/image')
    axes[2].grid(True)
    plt.tight_layout()
    plot_path = OUTPUT / 'fp32_convergence.png'
    fig.savefig(plot_path, dpi=160)
    print('Saved plot:', plot_path)
    plt.show()

# 1) Tải checkpoint cũ, gồm cả checkpoint QAT nếu dataset đã lưu.
copy_dataset_files_to_output()

# 2) Train thêm FP32 nếu chưa hội tụ/chưa đạt max epoch.
fp32_last = OUTPUT / 'fp32_last.pt'
fp32_epoch = checkpoint_epoch(fp32_last)
print(f'Current FP32 epoch: {fp32_epoch}/{FP32_MAX_EPOCHS}')
print('GPU count:', torch.cuda.device_count(), '| expected DDP when >=2 GPUs')

extra_done = 0
while fp32_epoch < FP32_MAX_EPOCHS and extra_done < FP32_MAX_EXTRA_EPOCHS_PER_RUN:
    history = read_json(FP32_HISTORY_JSON, [])
    converged, reason = fp32_convergence_status(history, FP32_PATIENCE, FP32_MIN_DELTA)
    if converged:
        print('FP32 đã hội tụ; bỏ qua train FP32:', reason)
        break
    print(f'===== FP32 epoch {fp32_epoch + 1}/{FP32_MAX_EPOCHS} =====', flush=True)
    run_fp32_one_epoch()
    extra_done += 1
    fp32_epoch = checkpoint_epoch(fp32_last)
    snapshot = OUTPUT / f'fp32_epoch_{fp32_epoch:02d}.pt'
    shutil.copy2(fp32_last, snapshot)
    history = update_fp32_history(fp32_epoch)
    converged, reason = fp32_convergence_status(history, FP32_PATIENCE, FP32_MIN_DELTA)
    status = {
        'converged': bool(converged),
        'reason': reason,
        'epoch': int(fp32_epoch),
        'max_epochs': int(FP32_MAX_EPOCHS),
        'patience': int(FP32_PATIENCE),
        'min_delta': float(FP32_MIN_DELTA),
    }
    FP32_CONVERGED_FLAG.write_text(json.dumps(status, indent=2), encoding='utf-8')
    print('Saved FP32 snapshot:', snapshot)
    print('Convergence:', status)
    upload_dataset(f'Pascal VOC FP32 epoch {fp32_epoch}, benchmark {BENCHMARK_IMAGES} images')

if fp32_last.exists():
    history = update_fp32_history(checkpoint_epoch(fp32_last))
    converged, reason = fp32_convergence_status(history, FP32_PATIENCE, FP32_MIN_DELTA)
    status = {
        'converged': bool(converged),
        'reason': reason,
        'epoch': int(checkpoint_epoch(fp32_last)),
        'max_epochs': int(FP32_MAX_EPOCHS),
        'patience': int(FP32_PATIENCE),
        'min_delta': float(FP32_MIN_DELTA),
    }
    FP32_CONVERGED_FLAG.write_text(json.dumps(status, indent=2), encoding='utf-8')
    plot_fp32_history()

print_checkpoint_summary(OUTPUT)


In [ ]:
# Cell 8 - Vẽ biểu đồ hội tụ FP32 từ history đã lưu
import json
from pathlib import Path

import matplotlib.pyplot as plt

history_path = OUTPUT / 'fp32_convergence_history.json'
assert history_path.exists(), 'Chưa có fp32_convergence_history.json; hãy chạy cell train/resume FP32 trước'
history = json.loads(history_path.read_text())
epochs = [item['epoch'] for item in history]
map_50_95 = [item.get('map_50_95') for item in history]
map_50 = [item.get('map_50') for item in history]
losses = [item.get('train_loss') for item in history]
latencies = [item.get('benchmark_latency_ms') for item in history]

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
axes[0].plot(epochs, map_50_95, marker='o', label='mAP@50:95')
axes[0].plot(epochs, map_50, marker='o', label='mAP@50')
axes[0].set_title('FP32 validation mAP')
axes[0].set_xlabel('Epoch')
axes[0].grid(True)
axes[0].legend()
axes[1].plot(epochs, losses, marker='o', color='tab:red')
axes[1].set_title('FP32 train loss')
axes[1].set_xlabel('Epoch')
axes[1].grid(True)
axes[2].plot(epochs, latencies, marker='o', color='tab:green')
axes[2].set_title(f'FP32 benchmark latency ({BENCHMARK_IMAGES} ảnh)')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('ms/image')
axes[2].grid(True)
plt.tight_layout()
PLOT_PATH = OUTPUT / 'fp32_convergence.png'
fig.savefig(PLOT_PATH, dpi=160)
print('Saved plot:', PLOT_PATH)
status_path = OUTPUT / 'fp32_converged.json'
if status_path.exists():
    print(json.dumps(json.loads(status_path.read_text()), indent=2))
plt.show()


In [ ]:
# Cell 9 - Resume/train QAT bù lỗi từng epoch, upload mỗi epoch
# Cell này chỉ train/resume QAT và lưu checkpoint. Convert TensorRT + benchmark nằm ở các cell sau.
import json
import shutil
from pathlib import Path

import kagglehub
import torch

QAT_HISTORY_JSON = OUTPUT / 'qat_train_history.json'

CHECKPOINT_INPUT_ROOT = Path('/kaggle/input/datasets/nguyenducthangtb/echteai-pascal-voc-convnext-qat')


def copy_checkpoint_from_dataset_if_missing(filename, required=False):
    """Copy checkpoint tu Kaggle Dataset ve OUTPUT neu OUTPUT chua co file."""
    dst = OUTPUT / filename
    if dst.exists():
        print(f'Using local {filename}:', dst)
        return dst

    candidates = [
        CHECKPOINT_INPUT_ROOT / filename,
        Path('/kaggle/input/echteai-pascal-voc-convnext-qat') / filename,
    ]
    candidates += sorted(Path('/kaggle/input').rglob(filename), key=lambda path: str(path))

    for src_path in candidates:
        if src_path.exists():
            OUTPUT.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src_path, dst)
            print(f'Copied {filename}: {src_path} -> {dst}')
            return dst

    if required:
        raise FileNotFoundError(
            f'Không tìm thấy {filename} trong {CHECKPOINT_INPUT_ROOT} hoặc /kaggle/input. '
            'Hãy attach dataset nguyenducthangtb/echteai-pascal-voc-convnext-qat vào notebook.'
        )
    print(f'Optional checkpoint not found: {filename}')
    return None


# Bootstrap checkpoint từ dataset đã lưu. Không overwrite file local nếu đã train tiếp trong session này.
copy_checkpoint_from_dataset_if_missing('fp32_best.pt', required=True)
copy_checkpoint_from_dataset_if_missing('fp32_last.pt', required=False)
copy_checkpoint_from_dataset_if_missing('qat_last.pt', required=False)
copy_checkpoint_from_dataset_if_missing('qat_best.pt', required=False)



def read_json(path, default):
    path = Path(path)
    return json.loads(path.read_text()) if path.exists() else default


def read_checkpoint_metrics(path):
    path = Path(path)
    if not path.exists():
        return {}
    payload = torch.load(path, map_location='cpu', weights_only=False)
    return payload.get('metrics', {}) if isinstance(payload, dict) else {}


def find_qat_resume_checkpoint(output_dir):
    """Ưu tiên qat_last.pt; nếu không có thì tìm qat_epoch_*.pt mới nhất."""
    output_dir = Path(output_dir)
    qat_last = output_dir / 'qat_last.pt'
    if qat_last.exists():
        return checkpoint_epoch(qat_last), qat_last

    scored = []
    for path in output_dir.glob('qat_epoch_*.pt'):
        try:
            scored.append((checkpoint_epoch(path), path))
        except Exception:
            pass
    if not scored:
        return 0, None
    scored.sort(key=lambda item: (item[0], str(item[1])))
    return scored[-1]


def update_qat_train_history(qat_epoch, input_checkpoint, qat_checkpoint):
    metrics = read_checkpoint_metrics(qat_checkpoint)
    train = metrics.get('train', {})
    record = {
        'qat_epoch': int(qat_epoch),
        'input_checkpoint': str(input_checkpoint) if input_checkpoint else str(OUTPUT / 'fp32_best.pt'),
        'qat_checkpoint': str(qat_checkpoint),
        'qat_train_loss': train.get('loss'),
        'global_steps': train.get('global_steps'),
        'steps_per_rank': train.get('steps_per_rank'),
        'seconds_avg_rank': train.get('seconds_avg_rank'),
        'map_50_95': metrics.get('map_50_95'),
        'map_50': metrics.get('map_50'),
        'precision': metrics.get('precision'),
        'recall': metrics.get('recall'),
        'accuracy': metrics.get('accuracy'),
        'mean_iou': metrics.get('mean_iou'),
        'benchmark': metrics.get('benchmark'),
    }
    history = read_json(QAT_HISTORY_JSON, [])
    history = [item for item in history if int(item.get('qat_epoch', -1)) != int(qat_epoch)]
    history.append(record)
    history = sorted(history, key=lambda item: item['qat_epoch'])
    QAT_HISTORY_JSON.write_text(json.dumps(history, indent=2, allow_nan=True), encoding='utf-8')
    return history


def upload_dataset(version_notes):
    if not UPLOAD_EVERY_EPOCH:
        print('UPLOAD_EVERY_EPOCH=False; bỏ qua upload.')
        return
    print('Uploading to:', CHECKPOINT_DATASET)
    print_checkpoint_summary(OUTPUT)
    kagglehub.dataset_upload(CHECKPOINT_DATASET, str(OUTPUT), version_notes=version_notes)
    print('Uploaded:', f'https://www.kaggle.com/datasets/{CHECKPOINT_DATASET}')

# Đảm bảo đã có FP32 checkpoint từ dataset hoặc từ cell FP32.
assert (OUTPUT / 'fp32_best.pt').exists(), 'Thiếu fp32_best.pt; hãy chạy cell resume/train FP32 trước'
status_path = OUTPUT / 'fp32_converged.json'
if status_path.exists():
    status = json.loads(status_path.read_text())
    fp32_ready = bool(status.get('converged')) or int(status.get('epoch', 0)) >= int(status.get('max_epochs', FP32_MAX_EPOCHS))
    assert fp32_ready or FORCE_START_QAT, f'FP32 chưa hội tụ: {status}. Đặt FORCE_START_QAT=True nếu vẫn muốn chạy QAT.'

for _ in range(QAT_MAX_EXTRA_EPOCHS_PER_RUN):
    current_qat_epoch, resume_checkpoint = find_qat_resume_checkpoint(OUTPUT)

    print('\n==============================')
    print('Current QAT epoch:', current_qat_epoch)
    print('Resume checkpoint:', resume_checkpoint)

    if current_qat_epoch >= QAT_TOTAL_EPOCHS:
        print(f'QAT đã đạt {current_qat_epoch}/{QAT_TOTAL_EPOCHS}; dừng.')
        break

    next_qat_epoch = current_qat_epoch + 1
    input_checkpoint = resume_checkpoint if current_qat_epoch > 0 else OUTPUT / 'fp32_best.pt'
    print(f'QAT epoch {next_qat_epoch}: input = {input_checkpoint}')

    if torch.cuda.device_count() >= 2:
        command = [
            sys.executable, '-m', 'torch.distributed.run', '--standalone', '--nproc_per_node=2',
            'scripts/train_qat_ddp.py',
            '--config', str(RUNTIME_CONFIG),
            '--fp32-checkpoint', str(OUTPUT / 'fp32_best.pt'),
            '--variant', VARIANT,
            '--epochs-this-run', '1',
            '--no-find-unused-parameters',
        ]
        if current_qat_epoch > 0:
            command += ['--resume', str(input_checkpoint)]
    else:
        command = [
            sys.executable, '-u', 'scripts/train_qat.py',
            '--config', str(RUNTIME_CONFIG),
            '--fp32-checkpoint', str(OUTPUT / 'fp32_best.pt'),
            '--variant', VARIANT,
            '--epochs-this-run', '1',
        ]
        if current_qat_epoch > 0:
            command += ['--resume', str(input_checkpoint)]
    if TRAIN_LIMIT is not None:
        command += ['--limit', str(TRAIN_LIMIT)]

    run_and_log(command, LOGS / f'qat_epoch_{next_qat_epoch:02d}.log', cwd=REPO)

    actual_epoch = checkpoint_epoch(OUTPUT / 'qat_last.pt')
    qat_snapshot = OUTPUT / f'qat_epoch_{actual_epoch:02d}.pt'
    shutil.copy2(OUTPUT / 'qat_last.pt', qat_snapshot)
    if (OUTPUT / 'qat_best.pt').exists():
        shutil.copy2(OUTPUT / 'qat_best.pt', OUTPUT / f'qat_best_epoch_{actual_epoch:02d}.pt')

    history = update_qat_train_history(actual_epoch, input_checkpoint, qat_snapshot)
    (OUTPUT / 'qat_train_status.json').write_text(json.dumps({
        'epoch': int(actual_epoch),
        'max_epochs': int(QAT_TOTAL_EPOCHS),
        'history_json': str(QAT_HISTORY_JSON),
        'last_checkpoint': str(OUTPUT / 'qat_last.pt'),
        'snapshot': str(qat_snapshot),
    }, indent=2), encoding='utf-8')

    print('Updated QAT train history:', QAT_HISTORY_JSON)
    print('Saved QAT snapshot:', qat_snapshot)
    upload_dataset(f'Pascal VOC QAT epoch {actual_epoch}, checkpoint only')

print('QAT checkpoint summary:')
print_checkpoint_summary(OUTPUT)


In [ ]:
# Cell 10 - Kiểm tra TensorRT và export ONNX FP32/QAT cho TensorRT
# Lưu ý: train vẫn là PyTorch. TensorRT chỉ bắt đầu từ export ONNX -> build .engine -> benchmark GPU hybrid.
import json
import subprocess
import sys
from pathlib import Path

import torch

try:
    import tensorrt as trt
    print('TensorRT:', trt.__version__)
except Exception as error:
    raise RuntimeError(
        'Môi trường hiện chưa import được TensorRT. '
        'Bật INSTALL_TENSORRT_IF_MISSING=True ở Cell 1, chạy Cell 2 để cài, restart runtime nếu cần, rồi chạy lại.'
    ) from error

TRT_DIR = OUTPUT / 'tensorrt_artifacts'
TRT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT.mkdir(parents=True, exist_ok=True)

CHECKPOINT_INPUT_ROOT = Path('/kaggle/input/datasets/nguyenducthangtb/echteai-pascal-voc-convnext-qat')


def resolve_checkpoint_from_dataset(filename, required=True):
    """Ưu tiên OUTPUT, sau đó lấy đúng checkpoint từ Kaggle Dataset đã attach."""
    local_path = OUTPUT / filename
    if local_path.exists():
        return local_path

    candidates = [
        CHECKPOINT_INPUT_ROOT / filename,
        Path('/kaggle/input/echteai-pascal-voc-convnext-qat') / filename,
    ]
    candidates += sorted(Path('/kaggle/input').rglob(filename), key=lambda path: str(path))

    for src_path in candidates:
        if src_path.exists():
            shutil.copy2(src_path, local_path)
            print(f'Copied {filename}: {src_path} -> {local_path}')
            return local_path

    if required:
        raise FileNotFoundError(
            f'Không tìm thấy {filename}. Dataset cần attach: '
            'nguyenducthangtb/echteai-pascal-voc-convnext-qat'
        )
    return None


FP32_CKPT = resolve_checkpoint_from_dataset('fp32_best.pt', required=True)
QAT_CKPT = resolve_checkpoint_from_dataset('qat_last.pt', required=True)
FP32_ONNX = TRT_DIR / 'convnext_fp32.onnx'
INT8_ONNX = TRT_DIR / 'convnext_qat_int8_qdq.onnx'

print('Checkpoint dataset root:', CHECKPOINT_INPUT_ROOT)
print('Using FP32 checkpoint:', FP32_CKPT)
print('Using QAT checkpoint:', QAT_CKPT)
assert FP32_CKPT.exists(), FP32_CKPT
assert QAT_CKPT.exists(), QAT_CKPT

common_shape = [
    '--height', str(TENSORRT_HEIGHT),
    '--width', str(TENSORRT_WIDTH),
    '--batch-size', str(TENSORRT_BATCH_SIZE),
]

command = [
    sys.executable, '-u', 'scripts/export_convnext_tensorrt_onnx.py',
    '--config', str(RUNTIME_CONFIG),
    '--model', 'fp32',
    '--fp32-checkpoint', str(FP32_CKPT),
    '--output', str(FP32_ONNX),
    *common_shape,
]
run_and_log(command, LOGS / 'export_fp32_onnx.log', cwd=REPO)

command = [
    sys.executable, '-u', 'scripts/export_convnext_tensorrt_onnx.py',
    '--config', str(RUNTIME_CONFIG),
    '--model', 'qat_graph',
    '--qat-checkpoint', str(QAT_CKPT),
    '--output', str(INT8_ONNX),
    '--tensorrt-friendly-int8',
    *common_shape,
]
run_and_log(command, LOGS / 'export_int8_qdq_onnx.log', cwd=REPO)

print('FP32 ONNX:', FP32_ONNX)
print('INT8 Q/DQ ONNX:', INT8_ONNX)


In [ ]:
# Cell 11 - Build TensorRT FP32/INT8 engine từ ONNX
FP32_ENGINE = TRT_DIR / 'convnext_fp32.engine'
INT8_ENGINE = TRT_DIR / 'convnext_int8.engine'

command = [
    sys.executable, '-u', 'scripts/build_tensorrt_engine.py',
    '--onnx', str(FP32_ONNX),
    '--engine', str(FP32_ENGINE),
    '--precision', 'fp32',
    '--workspace-mb', str(TENSORRT_WORKSPACE_MB),
]
run_and_log(command, LOGS / 'build_fp32_engine.log', cwd=REPO)

command = [
    sys.executable, '-u', 'scripts/build_tensorrt_engine.py',
    '--onnx', str(INT8_ONNX),
    '--engine', str(INT8_ENGINE),
    '--precision', 'int8',
    '--workspace-mb', str(TENSORRT_WORKSPACE_MB),
]
run_and_log(command, LOGS / 'build_int8_engine.log', cwd=REPO)

print('FP32 TensorRT engine:', FP32_ENGINE)
print('INT8 TensorRT engine:', INT8_ENGINE)


In [ ]:
# Cell 12 - Benchmark GPU: FP32 PyTorch checkpoint vs INT8 TensorRT hybrid, có đủ mAP + latency
# FP32: PyTorch CUDA, nhưng ở notebook này sẽ nạp từ qat_last.pt để so công bằng với nhánh INT8.
# INT8: TensorRT engine sinh từ đúng QAT checkpoint đó, FPN/RPN/ROI còn lại vẫn là PyTorch CUDA.
import json
import time
from pathlib import Path

import torch

from pipelines.convnext_qat.checkpoint import load_checkpoint
from pipelines.convnext_qat.config import load_config, quantized_modules_for_variant
from pipelines.convnext_qat.data import build_coco_loader, unwrap_coco_dataset
from pipelines.convnext_qat.metrics import _coco_metrics, native_detection_metrics
from pipelines.convnext_qat.models import build_fasterrcnn_convnext
from pipelines.convnext_qat.quantization import prepare_selective_qat, set_qat_phase
from scripts.benchmark_convnext_tensorrt_hybrid import TensorRTBackboneRunner, evaluate_hybrid_model

DEVICE = torch.device('cuda')
RESULT_JSON = OUTPUT / f'fp32_pytorch_gpu_vs_int8_tensorrt_m3_hybrid_{TENSORRT_BENCHMARK_TAG}.json'

assert torch.cuda.is_available(), 'CUDA GPU chưa bật'
assert QAT_CKPT.exists(), QAT_CKPT
assert INT8_ENGINE.exists(), INT8_ENGINE

config = load_config(str(RUNTIME_CONFIG), require_dataset=True)
loader = build_coco_loader(config, 'val', batch_size=1, shuffle=False, limit=TENSORRT_BENCHMARK_IMAGES)
ACTUAL_TENSORRT_BENCHMARK_IMAGES = len(loader.dataset)
print('TensorRT benchmark images:', ACTUAL_TENSORRT_BENCHMARK_IMAGES, '(full val)' if TENSORRT_BENCHMARK_IMAGES is None else f'(limit={TENSORRT_BENCHMARK_IMAGES})')

print('Loading QAT checkpoint:', QAT_CKPT)
qat_payload = torch.load(QAT_CKPT, map_location='cpu', weights_only=False)
metadata = qat_payload.get('extra', {}) if isinstance(qat_payload, dict) else {}
variant = str(metadata.get('variant', config['quantization'].get('variant', VARIANT))).upper()
backend = metadata.get('backend', config['quantization'].get('backend', 'auto'))
quantized_modules = metadata.get('quantized_modules', quantized_modules_for_variant(config, variant))
qat_model = build_fasterrcnn_convnext(config)
qat_model = prepare_selective_qat(qat_model, variant, backend, quantized_modules=quantized_modules)
load_checkpoint(QAT_CKPT, qat_model, map_location='cpu', strict=True)
set_qat_phase(qat_model, 'frozen')
qat_model.to(DEVICE).eval()

print('Loading INT8 TensorRT engine:', INT8_ENGINE)
int8_runner = TensorRTBackboneRunner(INT8_ENGINE)


@torch.inference_mode()
def evaluate_fp32_pytorch_with_timing(model, loader, device, warmup_images=10, progress_frequency=10):
    """Chạy nhánh PyTorch CUDA, vừa đo latency vừa gom prediction để tính mAP."""
    model.eval()
    predictions, targets, timings = [], [], []
    total_images = len(loader.dataset)
    processed = 0
    print(f'FP32 PyTorch evaluation started: target={total_images} images device={device}')

    for images, batch_targets in loader:
        cuda_images = [image.to(device) for image in images]

        torch.cuda.synchronize(device)
        started = time.perf_counter()
        outputs = model(cuda_images)
        torch.cuda.synchronize(device)

        elapsed_ms = (time.perf_counter() - started) * 1000.0 / max(len(images), 1)
        processed += len(images)
        if processed > warmup_images:
            timings.append(elapsed_ms)

        predictions.extend([{key: value.detach().cpu() for key, value in output.items()} for output in outputs])
        targets.extend([
            {key: value.detach().cpu() if torch.is_tensor(value) else value for key, value in target.items()}
            for target in batch_targets
        ])

        if progress_frequency and (processed == 1 or processed % progress_frequency == 0 or processed >= total_images):
            print(f'FP32 progress: {processed}/{total_images} images')

    print('FP32 inference completed; calculating detection metrics')
    metrics = native_detection_metrics(predictions, targets)
    dataset = unwrap_coco_dataset(loader.dataset)
    canonical = _coco_metrics(predictions, targets, dataset)
    if canonical:
        metrics.update(canonical)

    avg_ms = sum(timings) / max(len(timings), 1)
    metrics.update({
        'images': int(processed),
        'warmup_images': int(warmup_images),
        'measured_images': max(int(processed) - int(warmup_images), 0),
        'avg_inference_ms_per_image': float(avg_ms),
        'fps': 1000.0 / avg_ms if avg_ms > 0 else None,
        'device': str(device),
        'backend': 'pytorch_cuda',
    })
    print('FP32 evaluation completed')
    return metrics


print('Benchmarking FP32 PyTorch GPU with mAP...')
fp32_metrics = evaluate_fp32_pytorch_with_timing(
    qat_model,
    loader,
    DEVICE,
    warmup_images=10,
    progress_frequency=10,
)

print('Benchmarking INT8 TensorRT backbone GPU with mAP...')
int8_metrics = evaluate_hybrid_model(
    qat_model,
    int8_runner,
    loader,
    DEVICE,
    TENSORRT_HEIGHT,
    TENSORRT_WIDTH,
    scope='backbone',
    progress_frequency=10,
)
int8_metrics['backend'] = 'tensorrt_int8_hybrid_cuda'

latency_speedup = fp32_metrics['avg_inference_ms_per_image'] / int8_metrics['avg_inference_ms_per_image']
fps_speedup = int8_metrics['fps'] / fp32_metrics['fps'] if fp32_metrics.get('fps') else None
metric_delta = {
    'map_50_95': int8_metrics.get('map_50_95') - fp32_metrics.get('map_50_95'),
    'map_50': int8_metrics.get('map_50') - fp32_metrics.get('map_50'),
    'precision': int8_metrics.get('precision') - fp32_metrics.get('precision'),
    'recall': int8_metrics.get('recall') - fp32_metrics.get('recall'),
    'accuracy': int8_metrics.get('accuracy') - fp32_metrics.get('accuracy'),
    'mean_iou': int8_metrics.get('mean_iou') - fp32_metrics.get('mean_iou'),
    'latency_ms': int8_metrics.get('avg_inference_ms_per_image') - fp32_metrics.get('avg_inference_ms_per_image'),
    'fps': int8_metrics.get('fps') - fp32_metrics.get('fps'),
    'latency_speedup': latency_speedup,
    'fps_speedup': fps_speedup,
}

result = {
    'comparison': 'fp32_pytorch_gpu_vs_int8_tensorrt_m3_hybrid_gpu',
    'qat_checkpoint': str(QAT_CKPT),
    'int8_engine': str(INT8_ENGINE),
    'images': int(ACTUAL_TENSORRT_BENCHMARK_IMAGES),
    'requested_limit': TENSORRT_BENCHMARK_IMAGES,
    'fp32': fp32_metrics,
    'int8_tensorrt_hybrid': int8_metrics,
    'delta_int8_minus_fp32': metric_delta,
    'speedup': latency_speedup,
}
RESULT_JSON.write_text(json.dumps(result, indent=2, allow_nan=True), encoding='utf-8')

print('\nFP32 PyTorch GPU:')
print(f"  mAP@50:95: {fp32_metrics.get('map_50_95'):.4f}")
print(f"  mAP@50:    {fp32_metrics.get('map_50'):.4f}")
print(f"  Precision: {fp32_metrics.get('precision'):.4f}")
print(f"  Recall:    {fp32_metrics.get('recall'):.4f}")
print(f"  Mean IoU:  {fp32_metrics.get('mean_iou'):.4f}")
print(f"  Latency:   {fp32_metrics.get('avg_inference_ms_per_image'):.2f} ms/image")
print(f"  FPS:       {fp32_metrics.get('fps'):.2f}")

print('\nINT8 TensorRT backbone GPU:')
print(f"  mAP@50:95: {int8_metrics.get('map_50_95'):.4f}")
print(f"  mAP@50:    {int8_metrics.get('map_50'):.4f}")
print(f"  Precision: {int8_metrics.get('precision'):.4f}")
print(f"  Recall:    {int8_metrics.get('recall'):.4f}")
print(f"  Mean IoU:  {int8_metrics.get('mean_iou'):.4f}")
print(f"  Latency:   {int8_metrics.get('avg_inference_ms_per_image'):.2f} ms/image")
print(f"  FPS:       {int8_metrics.get('fps'):.2f}")

print('\nDelta INT8 - FP32:')
print(f"  mAP@50:95 delta: {metric_delta['map_50_95']:.4f}")
print(f"  mAP@50 delta:    {metric_delta['map_50']:.4f}")
print(f"  Latency delta:   {metric_delta['latency_ms']:.2f} ms/image")
print(f"  Speedup:         {latency_speedup:.4f}x")
print('Saved:', RESULT_JSON)

# Cập nhật history QAT với accuracy + tốc độ TensorRT nếu có.
history_path = OUTPUT / 'qat_convergence_history.json'
if history_path.exists():
    history = json.loads(history_path.read_text())
    qat_epoch = checkpoint_epoch(QAT_CKPT)
    for item in history:
        if int(item.get('qat_epoch', -1)) == int(qat_epoch):
            item['tensorrt_benchmark_json'] = str(RESULT_JSON)
            item['fp32_gpu_map_50_95'] = fp32_metrics.get('map_50_95')
            item['int8_tensorrt_map_50_95'] = int8_metrics.get('map_50_95')
            item['tensorrt_map_delta'] = metric_delta['map_50_95']
            item['fp32_gpu_latency_ms'] = fp32_metrics['avg_inference_ms_per_image']
            item['int8_tensorrt_latency_ms'] = int8_metrics['avg_inference_ms_per_image']
            item['tensorrt_speedup'] = latency_speedup
    history_path.write_text(json.dumps(history, indent=2, allow_nan=True), encoding='utf-8')


In [ ]:
# Cell 13 - Tổng hợp riêng kết quả FP32 PyTorch GPU vs INT8 TensorRT backbone GPU
import json
from pathlib import Path

import matplotlib.pyplot as plt

TRT_RESULT_JSON = OUTPUT / f'fp32_pytorch_gpu_vs_int8_tensorrt_m3_hybrid_{TENSORRT_BENCHMARK_TAG}.json'
assert TRT_RESULT_JSON.exists(), f'Chưa có TensorRT hybrid benchmark JSON: {TRT_RESULT_JSON}'

result = json.loads(TRT_RESULT_JSON.read_text())
fp32 = result['fp32']
int8 = result['int8_tensorrt_hybrid']
delta = result.get('delta_int8_minus_fp32', {})

summary = {
    'comparison': result.get('comparison'),
    'images': result.get('images'),
    'qat_checkpoint': result.get('qat_checkpoint'),
    'int8_engine': result.get('int8_engine'),
    'fp32_pytorch_gpu': {
        'map_50_95': fp32.get('map_50_95'),
        'map_50': fp32.get('map_50'),
        'precision': fp32.get('precision'),
        'recall': fp32.get('recall'),
        'accuracy': fp32.get('accuracy'),
        'mean_iou': fp32.get('mean_iou'),
        'f1': fp32.get('f1'),
        'latency_ms_per_image': fp32.get('avg_inference_ms_per_image'),
        'fps': fp32.get('fps'),
        'backend': fp32.get('backend'),
        'device': fp32.get('device'),
    },
    'int8_tensorrt_hybrid_gpu': {
        'map_50_95': int8.get('map_50_95'),
        'map_50': int8.get('map_50'),
        'precision': int8.get('precision'),
        'recall': int8.get('recall'),
        'accuracy': int8.get('accuracy'),
        'mean_iou': int8.get('mean_iou'),
        'f1': int8.get('f1'),
        'latency_ms_per_image': int8.get('avg_inference_ms_per_image'),
        'fps': int8.get('fps'),
        'backend': int8.get('backend'),
        'engine_shape': int8.get('engine_shape'),
        'scope': int8.get('scope'),
    },
    'delta_int8_minus_fp32': {
        'map_50_95': delta.get('map_50_95'),
        'map_50': delta.get('map_50'),
        'precision': delta.get('precision'),
        'recall': delta.get('recall'),
        'accuracy': delta.get('accuracy'),
        'mean_iou': delta.get('mean_iou'),
        'latency_ms': delta.get('latency_ms'),
        'fps': delta.get('fps'),
        'latency_speedup': delta.get('latency_speedup', result.get('speedup')),
        'fps_speedup': delta.get('fps_speedup'),
    },
}

SUMMARY_JSON = OUTPUT / 'pascal_voc_fp32_gpu_vs_int8_tensorrt_summary.json'
SUMMARY_JSON.write_text(json.dumps(summary, indent=2, allow_nan=True), encoding='utf-8')

print('FP32 PyTorch GPU vs INT8 TensorRT backbone GPU')
print('=' * 60)
print(f"Images: {summary['images']}")
print(f"QAT checkpoint:  {summary['qat_checkpoint']}")
print(f"INT8 engine:     {summary['int8_engine']}")
print()
print(f"{'Metric':<24} {'FP32 PyTorch GPU':>18} {'INT8 TRT hybrid':>18} {'Delta':>12}")
print('-' * 76)
for key, label in [
    ('map_50_95', 'mAP@50:95'),
    ('map_50', 'mAP@50'),
    ('precision', 'Precision'),
    ('recall', 'Recall'),
    ('accuracy', 'Accuracy'),
    ('mean_iou', 'Mean IoU'),
    ('f1', 'F1'),
]:
    fp32_value = summary['fp32_pytorch_gpu'].get(key)
    int8_value = summary['int8_tensorrt_hybrid_gpu'].get(key)
    diff = None if fp32_value is None or int8_value is None else int8_value - fp32_value
    print(f"{label:<24} {fp32_value:>18.4f} {int8_value:>18.4f} {diff:>12.4f}")

fp32_latency = summary['fp32_pytorch_gpu']['latency_ms_per_image']
int8_latency = summary['int8_tensorrt_hybrid_gpu']['latency_ms_per_image']
fp32_fps = summary['fp32_pytorch_gpu']['fps']
int8_fps = summary['int8_tensorrt_hybrid_gpu']['fps']
speedup = summary['delta_int8_minus_fp32']['latency_speedup']

print('-' * 76)
print(f"{'Latency ms/image':<24} {fp32_latency:>18.2f} {int8_latency:>18.2f} {int8_latency - fp32_latency:>12.2f}")
print(f"{'FPS':<24} {fp32_fps:>18.2f} {int8_fps:>18.2f} {int8_fps - fp32_fps:>12.2f}")
print(f"{'Speedup':<24} {'1.0000x':>18} {speedup:>17.4f}x {'':>12}")

labels = ['FP32 PyTorch GPU', 'INT8 TensorRT hybrid']
map_values = [summary['fp32_pytorch_gpu']['map_50_95'], summary['int8_tensorrt_hybrid_gpu']['map_50_95']]
latency_values = [fp32_latency, int8_latency]
fps_values = [fp32_fps, int8_fps]

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
axes[0].bar(labels, map_values, color=['tab:blue', 'tab:green'])
axes[0].set_title('mAP@50:95')
axes[0].set_ylim(0, max(map_values + [0.01]) * 1.15)
axes[0].grid(axis='y')
axes[0].tick_params(axis='x', rotation=10)

axes[1].bar(labels, latency_values, color=['tab:blue', 'tab:green'])
axes[1].set_title('Latency ms/image lower is better')
axes[1].grid(axis='y')
axes[1].tick_params(axis='x', rotation=10)

axes[2].bar(labels, fps_values, color=['tab:blue', 'tab:green'])
axes[2].set_title('FPS higher is better')
axes[2].grid(axis='y')
axes[2].tick_params(axis='x', rotation=10)

plt.tight_layout()
PLOT_PATH = OUTPUT / 'fp32_gpu_vs_int8_tensorrt_summary.png'
fig.savefig(PLOT_PATH, dpi=160)
print('Saved plot:', PLOT_PATH)
plt.show()

print('Saved summary:', SUMMARY_JSON)
print_checkpoint_summary(OUTPUT)


In [ ]:
# Cell 14 - Upload checkpoint/benchmark thành Kaggle Dataset mới hoặc version mới
import kagglehub

print('Uploading to:', CHECKPOINT_DATASET)
print_checkpoint_summary(OUTPUT)
kagglehub.dataset_upload(
    CHECKPOINT_DATASET,
    str(OUTPUT),
    version_notes=f'Pascal VOC ConvNeXt FP32/QAT + TensorRT M1 backbone benchmark {TENSORRT_BENCHMARK_TAG}',
)
print('Uploaded:', CHECKPOINT_DATASET)
print('Link:', f'https://www.kaggle.com/datasets/{CHECKPOINT_DATASET}')


In [ ]:
# Cell 15 - Video visualization: FP32 bên trái, INT8 TensorRT hybrid bên phải
import base64
import json
import time
from pathlib import Path

import cv2
import numpy as np
import torch
import torch.nn.functional as F
from IPython.display import HTML, display
from torchvision.models.detection.image_list import ImageList
from torchvision.transforms.functional import to_tensor

from pipelines.convnext_qat.checkpoint import load_checkpoint
from pipelines.convnext_qat.compiler import resolve_compiler_scope
from pipelines.convnext_qat.config import load_config
from pipelines.convnext_qat.data import build_coco_loader, unwrap_coco_dataset
from pipelines.convnext_qat.models import build_fasterrcnn_convnext
from pipelines.convnext_qat.quantization import prepare_selective_qat, set_qat_phase
from scripts.benchmark_convnext_tensorrt_hybrid import (
    M3_SCOPE,
    TensorRTBackboneRunner,
    _features_from_fpn_tuple,
    _rpn_proposals_from_precomputed_head,
    _split_m3_outputs,
)

VIDEO_SOURCE = Path('/kaggle/input/datasets/nguyenducthangtb/video-test1/traffic_video.mp4')
VIDEO_OUTPUT = OUTPUT / f'pascalvoc_fp32_vs_int8_hybrid_{VIDEO_SOURCE.stem}.mp4'
VIDEO_JSON = VIDEO_OUTPUT.with_suffix('.json')
SCORE_THRESHOLD = 0.40
FRAME_STRIDE = 1
MAX_FRAMES = None
FIXED_HEIGHT = int(TENSORRT_HEIGHT)
FIXED_WIDTH = int(TENSORRT_WIDTH)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


def draw_info_box(frame, title, lines, color, box_width=500):
    canvas = frame.copy()
    h, w = canvas.shape[:2]
    line_h = 30
    box_h = 34 + line_h * len(lines) + 14
    x1 = max(0, w - box_width - 14)
    y1 = 14
    x2 = w - 10
    y2 = min(h - 10, y1 + box_h)
    overlay = canvas.copy()
    cv2.rectangle(overlay, (x1, y1), (x2, y2), (0, 0, 0), -1)
    cv2.addWeighted(overlay, 0.62, canvas, 0.38, 0, canvas)
    cv2.rectangle(canvas, (x1, y1), (x2, y2), color, 2)
    cv2.putText(canvas, title, (x1 + 14, y1 + 30), cv2.FONT_HERSHEY_SIMPLEX, 0.85, color, 2, cv2.LINE_AA)
    for idx, line in enumerate(lines, 1):
        cv2.putText(canvas, line, (x1 + 14, y1 + 30 + idx * line_h), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 255, 255), 1, cv2.LINE_AA)
    return canvas


@torch.inference_mode()
def preprocess_single_frame(model, image_tensor, device, fixed_height, fixed_width):
    mean = torch.tensor(model.transform.image_mean, device=device).view(-1, 1, 1)
    std = torch.tensor(model.transform.image_std, device=device).view(-1, 1, 1)
    image_tensor = image_tensor.to(device)
    resized = F.interpolate(((image_tensor - mean) / std).unsqueeze(0), size=(fixed_height, fixed_width), mode='bilinear', align_corners=False).squeeze(0)
    original_size = tuple(int(v) for v in image_tensor.shape[-2:])
    return ImageList(resized.unsqueeze(0), [(fixed_height, fixed_width)]), original_size


@torch.inference_mode()
def predict_fp32(model, image_tensor, device):
    if device.type == 'cuda':
        torch.cuda.reset_peak_memory_stats(device)
        torch.cuda.synchronize(device)
    started = time.perf_counter()
    pred = model([image_tensor.to(device)])[0]
    if device.type == 'cuda':
        torch.cuda.synchronize(device)
        vram_alloc_mb = torch.cuda.memory_allocated(device) / (1024 ** 2)
        vram_peak_mb = torch.cuda.max_memory_allocated(device) / (1024 ** 2)
    else:
        vram_alloc_mb = 0.0
        vram_peak_mb = 0.0
    return {k: v.detach().cpu() for k, v in pred.items()}, (time.perf_counter() - started) * 1000.0, vram_alloc_mb, vram_peak_mb


@torch.inference_mode()
def predict_hybrid(model, runner, image_tensor, device, scope, fixed_height, fixed_width):
    image_list, original_size = preprocess_single_frame(model, image_tensor, device, fixed_height, fixed_width)
    if device.type == 'cuda':
        torch.cuda.reset_peak_memory_stats(device)
        torch.cuda.synchronize(device)
    started = time.perf_counter()
    trt_outputs = runner(image_list.tensors)
    if scope == 'backbone':
        feature_dict = _features_from_fpn_tuple(trt_outputs)
        features = model.backbone.fpn(feature_dict)
        proposals, _ = model.rpn(image_list, features, None)
    elif scope == M3_SCOPE:
        fpn_features, shared_features, objectness = _split_m3_outputs(trt_outputs)
        features = _features_from_fpn_tuple(fpn_features)
        pred_bbox_deltas = [model.rpn.head.bbox_pred(feature) for feature in shared_features]
        proposals = _rpn_proposals_from_precomputed_head(model, image_list, features, objectness, pred_bbox_deltas)
    else:
        features = _features_from_fpn_tuple(trt_outputs)
        proposals, _ = model.rpn(image_list, features, None)
    outputs, _ = model.roi_heads(features, proposals, image_list.image_sizes, None)
    outputs = model.transform.postprocess(outputs, image_list.image_sizes, [original_size])
    if device.type == 'cuda':
        torch.cuda.synchronize(device)
        vram_alloc_mb = torch.cuda.memory_allocated(device) / (1024 ** 2)
        vram_peak_mb = torch.cuda.max_memory_allocated(device) / (1024 ** 2)
    else:
        vram_alloc_mb = 0.0
        vram_peak_mb = 0.0
    return {k: v.detach().cpu() for k, v in outputs[0].items()}, (time.perf_counter() - started) * 1000.0, vram_alloc_mb, vram_peak_mb


def draw_boxes(frame, prediction, title, latency_ms, threshold, color, label_names, extra_lines=None):
    canvas = frame.copy()
    boxes = prediction['boxes'].detach().cpu().numpy()
    scores = prediction['scores'].detach().cpu().numpy()
    labels = prediction['labels'].detach().cpu().numpy()
    kept = 0
    for box, score, label in zip(boxes, scores, labels):
        if float(score) < threshold:
            continue
        kept += 1
        x1, y1, x2, y2 = np.rint(box).astype(int)
        name = label_names.get(int(label), f'class_{int(label)}')
        cv2.rectangle(canvas, (x1, y1), (x2, y2), color, 2)
        text = f'{name} {float(score):.2f}'
        (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.48, 1)
        cv2.rectangle(canvas, (x1, max(0, y1 - th - 8)), (x1 + tw + 5, y1), color, -1)
        cv2.putText(canvas, text, (x1 + 2, max(th + 1, y1 - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.48, (0, 0, 0), 1, cv2.LINE_AA)
    info = [f'latency: {latency_ms:.1f} ms', f'fps: {1000.0 / latency_ms:.2f}', f'detections: {kept}', f'threshold: {threshold:.2f}']
    if extra_lines:
        info.extend(extra_lines)
    return draw_info_box(canvas, title, info, color), kept


FP32_CKPT = globals().get('FP32_CKPT', globals().get('FP32_CHECKPOINT'))
QAT_CKPT = globals().get('QAT_CKPT', globals().get('QAT_CHECKPOINT'))
assert FP32_CKPT is not None, 'Missing FP32 checkpoint variable'
assert QAT_CKPT is not None, 'Missing QAT checkpoint variable'
print('Video source:', VIDEO_SOURCE)
print('Video output:', VIDEO_OUTPUT)
print('FP32 checkpoint:', FP32_CKPT)
print('QAT checkpoint (for INT8 hybrid topology):', QAT_CKPT)

runtime = load_config(str(RUNTIME_CONFIG), require_dataset=True)
runtime['model']['pretrained_backbone'] = False
runtime['quantization']['compiler']['scope'] = resolve_compiler_scope(runtime)

print('Loading FP32 model...')
fp32_model = build_fasterrcnn_convnext(runtime)
load_checkpoint(FP32_CKPT, fp32_model, map_location='cpu', strict=True)
fp32_model = fp32_model.to(DEVICE).eval()

print('Loading QAT model for TensorRT hybrid...')
qat_payload = torch.load(QAT_CKPT, map_location='cpu', weights_only=False)
metadata = qat_payload.get('extra', {}) if isinstance(qat_payload, dict) else {}
variant = str(metadata.get('variant', runtime['quantization'].get('variant', 'M1'))).upper()
backend = metadata.get('backend', runtime['quantization'].get('backend', 'auto'))
quantized_modules = metadata.get('quantized_modules')
hybrid_model = build_fasterrcnn_convnext(runtime)
hybrid_model = prepare_selective_qat(hybrid_model, variant, backend, quantized_modules=quantized_modules)
load_checkpoint(QAT_CKPT, hybrid_model, map_location='cpu', strict=True)
set_qat_phase(hybrid_model, 'frozen')
hybrid_model = hybrid_model.to(DEVICE).eval()

print('Loading TensorRT engine:', INT8_ENGINE)
int8_runner = TensorRTBackboneRunner(INT8_ENGINE)

loader = build_coco_loader(runtime, 'val', shuffle=False, limit=1, batch_size=1)
dataset = unwrap_coco_dataset(loader.dataset)
label_names = dataset.label_to_name

if VIDEO_OUTPUT.exists():
    VIDEO_OUTPUT.unlink()
if VIDEO_JSON.exists():
    VIDEO_JSON.unlink()

capture = cv2.VideoCapture(str(VIDEO_SOURCE))
if not capture.isOpened():
    raise RuntimeError(f'Cannot open video: {VIDEO_SOURCE}')
input_fps = capture.get(cv2.CAP_PROP_FPS) or 30.0
output_width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH) or 0)
output_height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT) or 0)
writer = cv2.VideoWriter(str(VIDEO_OUTPUT), cv2.VideoWriter_fourcc(*'mp4v'), input_fps / max(FRAME_STRIDE, 1), (output_width * 2, output_height))
if not writer.isOpened():
    raise RuntimeError(f'Cannot create output video: {VIDEO_OUTPUT}')

fp32_times, int8_times = [], []
fp32_vrams, fp32_peaks, int8_vrams, int8_peaks = [], [], [], []
fp32_dets = int8_dets = 0
processed = frame_index = 0

try:
    while True:
        ok, frame = capture.read()
        if not ok or (MAX_FRAMES is not None and processed >= MAX_FRAMES):
            break
        if frame_index % FRAME_STRIDE:
            frame_index += 1
            continue
        frame_index += 1
        image_tensor = to_tensor(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        fp32_pred, fp32_ms, fp32_vram, fp32_peak = predict_fp32(fp32_model, image_tensor, DEVICE)
        int8_pred, int8_ms, int8_vram, int8_peak = predict_hybrid(
            hybrid_model,
            int8_runner,
            image_tensor,
            DEVICE,
            runtime['quantization']['compiler']['scope'],
            FIXED_HEIGHT,
            FIXED_WIDTH,
        )
        left, left_count = draw_boxes(
            frame,
            fp32_pred,
            'FP32 PyTorch CUDA',
            fp32_ms,
            SCORE_THRESHOLD,
            (66, 153, 245),
            label_names,
            extra_lines=[f'vram: {fp32_vram:.1f} MB | peak: {fp32_peak:.1f} MB'],
        )
        right, right_count = draw_boxes(
            frame,
            int8_pred,
            'INT8 TensorRT hybrid',
            int8_ms,
            SCORE_THRESHOLD,
            (249, 115, 22),
            label_names,
            extra_lines=[f'vram: {int8_vram:.1f} MB | peak: {int8_peak:.1f} MB'],
        )
        writer.write(np.concatenate([left, right], axis=1))
        fp32_times.append(fp32_ms)
        int8_times.append(int8_ms)
        fp32_vrams.append(fp32_vram)
        fp32_peaks.append(fp32_peak)
        int8_vrams.append(int8_vram)
        int8_peaks.append(int8_peak)
        fp32_dets += left_count
        int8_dets += right_count
        processed += 1
        if processed == 1 or processed % 20 == 0:
            print(f'frame={processed} fp32={fp32_ms:.1f}ms int8={int8_ms:.1f}ms speedup={fp32_ms / int8_ms:.3f}x', flush=True)
finally:
    capture.release()
    writer.release()

if not processed:
    raise RuntimeError('No video frames were processed')

summary = {
    'video': str(VIDEO_SOURCE),
    'output': str(VIDEO_OUTPUT),
    'processed_frames': processed,
    'frame_stride': FRAME_STRIDE,
    'max_frames': MAX_FRAMES,
    'score_threshold': SCORE_THRESHOLD,
    'tensorRT_shape': [FIXED_HEIGHT, FIXED_WIDTH],
    'fp32_pytorch_gpu': {
        'mean_latency_ms': float(np.mean(fp32_times)),
        'fps': 1000.0 / float(np.mean(fp32_times)),
        'detections': int(fp32_dets),
        'mean_vram_mb': float(np.mean(fp32_vrams)) if fp32_vrams else 0.0,
        'peak_vram_mb': float(np.max(fp32_peaks)) if fp32_peaks else 0.0,
    },
    'int8_tensorrt_hybrid_gpu': {
        'mean_latency_ms': float(np.mean(int8_times)),
        'fps': 1000.0 / float(np.mean(int8_times)),
        'detections': int(int8_dets),
        'mean_vram_mb': float(np.mean(int8_vrams)) if int8_vrams else 0.0,
        'peak_vram_mb': float(np.max(int8_peaks)) if int8_peaks else 0.0,
    },
    'speedup': float(np.mean(fp32_times) / np.mean(int8_times)),
}
VIDEO_JSON.write_text(json.dumps(summary, indent=2), encoding='utf-8')
print(json.dumps(summary, indent=2))
print('Saved video:', VIDEO_OUTPUT)
print('Saved metrics:', VIDEO_JSON)
with open(VIDEO_OUTPUT, 'rb') as f:
    video_b64 = base64.b64encode(f.read()).decode('utf-8')
display(HTML(f"""
<video width="1200" controls autoplay loop>
  <source src="data:video/mp4;base64,{video_b64}" type="video/mp4">
</video>
"""))
